첫번째 실습에서 우리는 풀 파인튜닝(Full Fine-tuning)이 뛰어난 정확도를 내지만, 모델의 모든 매개변수(100%)를 학습해야 하므로 컴퓨팅 자원과 저장공간 소모가 극심함을 배웠습니다.

이번 2번재 실습에서는 이러한 문제를 극복하기 위한 대표적인 **PEFT (Parameter-Efficient Fine-Tuning)** 기법인 **LoRA (Low-Rank Adaptation)** 의 개념을 배우고, 직접 모델에 적용하여 학습을 수행해 봅니다.

## 1. PEFT와 LoRA의 이해

### 1) PEFT (Parameter-Efficient Fine-Tuning) 란?
- 사전 훈련된 거대 모델(Base Model)의 대부분 가중치는 **동결(Freeze)** 하고, 매우 일부분의 매개변수만 추가하거나 튜닝하여 새로운 작업에 적응시키는 기법입니다.
- **장점**:
  - **VRAM 소모 급감**: 최적화 기구(Adam 등)가 관리해야 할 파라미터가 크게 줄어 GPU 메모리를 훨씬 적게 씁니다.
  - **저장용량 절약**: 전체 모델 파일(예: 300MB~수십GB)을 태스크마다 따로 저장할 필요 없이, 몇십 KB ~ 몇 MB 크기의 가중치(어댑터) 파일만 관리하면 됩니다.
  - **치명적 망각 방지**: 기존 파라미터를 그대로 보존하므로 일반 성능이 유지됩니다.

### 2) LoRA (Low-Rank Adaptation) 의 작동 원리
- 모델이 학습할 가중치 업데이트 행렬 $\Delta W$를 직접 훈련하는 대신, 저차원의 두 개의 행렬 $A$와 $B$의 곱으로 **분해(Decomposition)** 하여 학습시키는 기법입니다.
  $$\Delta W \approx B \cdot A$$
  - 여기서 원래 가중치가 $d \times k$ 차원이고 저차원 랭크가 $r$ ($r \ll d, k$) 일 때:
    - $B \in \mathbb{R}^{d \times r}$ (0으로 초기화)
    - $A \in \mathbb{R}^{r \times k}$ (가우시안 분포로 초기화)
- 결과적으로 가중치 매개변수의 수는 $(d \times k)$ 에서 $r \times (d + k)$ 로 압도적으로 줄어듭니다.

### 라이브러리

In [6]:
from peft import LoraConfig, get_peft_model, TaskType
# 1교시 라이브러리 추가
import time
import torch
import pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,Seq2SeqTrainingArguments,DataCollatorForSeq2Seq
    )
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'using devcies : {device}')

using devcies : cpu


### 데이터

In [7]:
# 1교시에 사용한 데이터 train, test
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

from glob import glob
import pandas as pd
import re
df = pd.read_csv(glob(path+'/*')[0])
df['review'] = df['review'].apply(lambda x : re.sub(r'[^가-힣a-zA-Z0-9\s]', '', x.strip().lower()))

df['sentiment'].apply(lambda x : x.strip().lower())
t = [{'text':review, 'label':sentiment} for review, sentiment in zip(df['review'].to_list(),df['sentiment'].to_list())]
train_data = t[:500]
test_data = t[-100:]
print(len(train_data), len(test_data))

Path to dataset files: C:\Users\Playdata\.cache\kagglehub\datasets\lakshmi25npathi\imdb-dataset-of-50k-movie-reviews\versions\1
500 100


## 베이스 모델 로드 및 LoRA 설정 적용
- `peft` 라이브러리의 `LoraConfig`를 선언하고 모델에 주입합니다.
- `target_modules` 인자에 튜닝을 진행할 어텐션 매개변수인 쿼리(`q`)와 밸류(`v`) 행렬을 명시합니다.

In [10]:
model_name = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to(device)

# LoRA 설정
peft_config= LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r = 8,          # 저차원 랭크의 크기
    lora_alpha=32,  # 가중치 스케일 상수
    lora_dropout=0.1,
    target_modules=['q','v']    
)

# PEFT 모델로 변환
model = get_peft_model(model, peft_config=peft_config)
# 파라메터 계산함수
def get_trainable_params(model):
    all_param = 0
    trainable_params= 0
    for _, param in model.named_parameters():
        all_param += param.numel()  # 파라메터 개수
        if param.requires_grad:
            trainable_params += param.numel()
    return all_param,trainable_params, trainable_params/all_param*100

all_p, train_p, pct = get_trainable_params(model)
all_p, train_p, pct

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


(77305216, 344064, 0.445072166928555)

### 데이터 토큰화 처리

In [11]:
# 1. 데이터 토크나이징 함수 - 전처리
def preprocess_fuction(example):
    # T5모델에 맞게 토큰화
    inputs = [ f"Review: {text}\nSentiment: Answer with either positive or negative."  for text in example['text'] ]
    model_input = tokenizer(inputs,max_length=128, truncation=True)
    # 라벨을 토큰화
    lables = tokenizer(text_target=example['label'],max_length=128, truncation=True)
    model_input['labels'] = lables['input_ids']
    return model_input
# 2. DataSet객체
from datasets import Dataset
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

# 3. 토큰화 매핑()
tokenized_train = train_dataset.map(preprocess_fuction,batched=True, remove_columns=train_dataset.column_names)
tokenized_test = test_dataset.map(preprocess_fuction,batched=True, remove_columns=train_dataset.column_names)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]